# 03 · Orchestrate — Medical-Information QA Assistant

**Vertical:** Healthcare · **Use case:** medical-information-qa · **ADLC stage:** Orchestrate

> Self-directed module · ~15 min. By the end you can:
> - give medical-information conversations thread memory with `+ checkpointer=MemorySaver()`
> - pause suspected adverse-event handling for medical-review approval with `HumanInTheLoopMiddleware`
> - layer PII redaction and policy blocking as middleware before model calls
> - open the same graph in **LangGraph Studio** and approve an interrupt by hand
>
> 🧭 **Builds on Labs 01–02; runs standalone.** Tools + prompt are Lab 01's (imported); what's shown
> here is runtime wiring — one new `create_agent` kwarg per capability.

![Architecture](images/03_orchestrate.png)

| What's new | How |
|---|---|
| Memory | `+ checkpointer=MemorySaver()` and a stable `thread_id` |
| Human-in-the-loop | `+ middleware=[HumanInTheLoopMiddleware(...)]` |
| PII + policy gates | `+ middleware=[PIIMiddleware(...), custom @before_model, ToolCallLimitMiddleware(...)]` |
| Studio | `medical_information_qa_graph.py` + `langgraph.json` |

## 0. Setup

Thin here — env, path shim, and active workspace are enough for a standalone orchestration lab.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # lab .env wins over a globally-set key/workspace
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))

print("workspace:", os.getenv("LANGSMITH_WORKSPACE_ID"), "| tracing:", os.getenv("LANGSMITH_TRACING"))

## 1. Build the baseline agent

The tools and prompt are Lab 01's plumbing. The `create_agent` call is the lesson, so it is shown here
and then repeated with one added kwarg for each capability.

In [ ]:
from langchain.agents import create_agent

from medical_information_qa_graph import MedicalInformationAnswer, PROMPT_NAME, PROMPT_PATH, medical_information_qa_tools
from utils.models import get_model
from utils.prompts import get_prompt

model = get_model()
tools = medical_information_qa_tools()  # get_inquiry, get_drug_profile, search_prescribing_info, check_safety_policy
system_prompt = get_prompt(PROMPT_NAME, fallback=PROMPT_PATH.read_text().strip())

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=MedicalInformationAnswer,  # typed answer from Lab 01; memory + HITL add to this
)

**The agent's shape** — `create_agent` returns a compiled LangGraph graph; render it before adding
runtime controls.

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## 2. Add memory with a checkpointer

A **checkpointer** persists conversation state by `thread_id`. The follow-up can ask about “that drug”
without restating the full inquiry.

In [ ]:
import uuid

from langgraph.checkpoint.memory import MemorySaver

mem_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=MedicalInformationAnswer,
    checkpointer=MemorySaver(),  # ← only addition vs §1
)
thread = {"configurable": {"thread_id": str(uuid.uuid4())}}  # one med-info conversation

> 💡 The graph **shape is unchanged** from §1 — a checkpointer adds persistence keyed by `thread_id`,
> not new nodes. Memory is runtime state, not structure.

In [ ]:
from utils.trace import ask  # .invoke + printed trace link

r1 = ask(mem_agent, "MIQ-005: What dosing and vaccine precautions are listed for Elmurestat?", config=thread)
print(f"{r1.disposition} · flags={r1.safety_flags}")

In [ ]:
r2 = ask(mem_agent, "For that drug, is a live vaccine precaution mentioned?", config=thread)  # same thread → remembers Elmurestat
print("answer:", r2.answer)
print("citations:", r2.citations)

## 3. Pause adverse-event review before policy lookup

Suspected adverse events should route to a medical-review workflow. We pause before `check_safety_policy`
so a reviewer can approve the policy/escalation step.

```text
invoke → ⏸️ interrupt → decide → Command(resume=…) → ▶️ continue
```

| Decision | Effect |
|---|---|
| ✅ `approve` | run the policy lookup as proposed |
| ✏️ `edit` | change the args, then run |
| ⛔ `reject` | skip it; tell the model why |

> 💡 HITL needs persistence to pause/resume — hence the checkpointer here. In Studio, managed
> persistence provides the same pause/resume behavior for `medical_information_qa_agent_hitl`.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

from utils.trace import invoke_traced  # .invoke + printed trace link, returns the full result

hitl_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=MedicalInformationAnswer,
    checkpointer=MemorySaver(),  # same as memory agent
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"check_safety_policy": True})],  # ← only addition vs memory agent
)
hitl_thread = {"configurable": {"thread_id": str(uuid.uuid4())}}

**The shape changed.** Unlike the checkpointer, HITL is *middleware* — so it adds review nodes where the
interrupt can fire.

In [ ]:
display(Image(hitl_agent.get_graph().draw_mermaid_png()))

**⏸️ Invoke → interrupt.** The run stops before the safety-policy lookup and returns the proposed tool
call. If the model answers without the tool, the cell still completes and prints the final result.

In [ ]:
result = invoke_traced(
    hitl_agent,
    {"messages": [{"role": "user", "content": "MIQ-009: [FAKE_NAME: Patient Alpha] had facial swelling and hives after Elmurestat and went to urgent care. Their fake phone is 000-000-0000."}]},
    config=hitl_thread,
)
if result.get("__interrupt__"):
    requests = result["__interrupt__"][0].value["action_requests"]  # requested tool calls
    for req in requests:
        print("⏸️  paused — wants:", req["name"], "| args:", req["args"])
else:
    final = result.get("structured_response")
    print("No interrupt fired; final disposition:", getattr(final, "disposition", result["messages"][-1].content[:200]))

**▶️ Decide → resume.** Approve until the run finishes; the model may request more than one policy lookup.

In [ ]:
approvals = 0
while result.get("__interrupt__"):
    pending = result["__interrupt__"][0].value["action_requests"]
    approvals += len(pending)
    decisions = [{"type": "approve"} for _ in pending]  # one decision per pending tool call
    result = hitl_agent.invoke(Command(resume={"decisions": decisions}), config=hitl_thread)
report = result.get("structured_response")
if report:
    print(f"✅ {approvals} approval(s) → {report.disposition}, AE escalate={report.escalate_adverse_event}")
    print("redactions:", report.redaction_notes)
else:
    print("completed without structured response:", result["messages"][-1].content[:500])

## 4. Guardrails — PII redaction + policy blocking middleware

Middleware composes: pass a **list** and each one wraps the agent. A production medical-information agent
should redact identifiers before model calls and short-circuit obviously unsafe requests.

| Guardrail | Middleware | Protects against |
|---|---|---|
| Redact emails / phone numbers | `PIIMiddleware(..., strategy="redact")` | PII leaking into model context or traces |
| Redact fixture placeholders | custom `PIIMiddleware("fake_identifier", detector=...)` | synthetic `[FAKE_NAME: ...]` identifiers |
| Block real-drug / emergency requests | custom `@before_model` (jumps to `end`) | real medical advice or emergency triage |
| Cap tool loops | `ToolCallLimitMiddleware(run_limit=8, exit_behavior="end")` | runaway cost / infinite loops |

**The custom policy gate** — a `@before_model` hook that reads the latest human message and short-circuits
to `end` for real-world medical or emergency requests. The lab only answers fictional fixture questions.

In [ ]:
from langchain.agents.middleware import PIIMiddleware, ToolCallLimitMiddleware, before_model
from langchain_core.messages import AIMessage, HumanMessage

_REAL_WORLD_OR_EMERGENCY = (
    "ozempic", "humira", "lipitor", "metformin", "my real", "911", "emergency", "chest pain now",
)


@before_model(can_jump_to=["end"])  # this hook may short-circuit straight to the end node
def block_real_world_medical_requests(state, runtime):
    human = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    text = (getattr(human, "content", "") or "").lower()
    if any(marker in text for marker in _REAL_WORLD_OR_EMERGENCY):
        return {
            "jump_to": "end",
            "messages": [AIMessage("⛔ Blocked: this lab handles only fictional PI fixtures. For real medical or emergency concerns, contact a qualified clinician or local emergency services.")],
        }
    return None  # otherwise fall through to the model

**Build the guarded agent** — the same call, now with a middleware stack.

In [ ]:
guarded_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=MedicalInformationAnswer,
    middleware=[
        PIIMiddleware("email", strategy="redact"),                         # scrub emails from the input
        PIIMiddleware("phone", detector=r"\b\d{3}-\d{3}-\d{4}\b", strategy="redact"),
        PIIMiddleware("fake_identifier", detector=r"\[FAKE_(?:NAME|EMAIL|PHONE|ADDRESS|ID):[^\]]+\]", strategy="redact"),
        block_real_world_medical_requests,                                  # ← custom policy gate
        ToolCallLimitMiddleware(run_limit=8, exit_behavior="end"),          # cap runaway tool loops
    ],
)

**The shape changed again.** Each middleware adds `before_model` / `after_model` nodes, so the stack
becomes a chain. We list the structure instead of rendering a crowded PNG.

In [ ]:
_g = guarded_agent.get_graph()
print("nodes:", [n for n in _g.nodes])
print("edges:")
for _e in _g.edges:
    print(f"  {_e.source} -> {_e.target}" + ("  [conditional]" if _e.conditional else ""))

**✅ A legitimate adverse-event request runs — and placeholder PII is redacted** before the model sees it.

In [ ]:
guarded = invoke_traced(guarded_agent, {"messages": [{"role": "user", "content": "MIQ-009: [FAKE_NAME: Patient Alpha] had facial swelling and hives after Elmurestat. Email reporter.alpha@example.invalid and phone 000-000-0000."}]})
stored = next(m for m in guarded["messages"] if isinstance(m, HumanMessage))
print("model saw:", stored.content)  # → placeholders / email / phone are redacted
if guarded.get("structured_response"):
    print("verdict  :", guarded["structured_response"].disposition)
    print("redacted :", guarded["structured_response"].pii_redacted)

**⛔ A real-world medical request is blocked** before the model runs — no tools, no fixture answer.

In [ ]:
blocked = invoke_traced(guarded_agent, {"messages": [{"role": "user", "content": "Can I take Ozempic with my real blood pressure medicine?"}]})
print(blocked["messages"][-1].content)
print("structured_response:", blocked.get("structured_response"))  # None — the guardrail ended the run early

## 5. Open it in LangGraph Studio

The same `create_agent` calls are packaged in `medical_information_qa_graph.py` and registered in
`langgraph.json`. From the repository root:

```bash
cd /path/to/content
uv run langgraph dev     # serves at http://127.0.0.1:2024
```

Studio: **https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024**

| Graph | Try in Studio |
|---|---|
| `medical_information_qa_agent` | *"MIQ-001: What is Alvexora indicated for and what is the starting dose?"* → cited PI answer |
| `medical_information_qa_agent_hitl` | *"MIQ-009: [FAKE_NAME: Patient Alpha] had facial swelling and hives after Elmurestat and went to urgent care."* → pauses before `check_safety_policy`; click **Approve** |

## 6. Recap

| Topic | API |
|---|---|
| Memory across inquiry turns | `create_agent(…, checkpointer=MemorySaver())` + `config={"configurable": {"thread_id": …}}` |
| Pause for medical-review approval | `create_agent(…, middleware=[HumanInTheLoopMiddleware(interrupt_on={...})])` |
| Resume an interrupt | `agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=...)` |
| PII redaction + policy gates | `PIIMiddleware(...)`, a custom `@before_model` block, `ToolCallLimitMiddleware(...)` |
| Studio / deploy | `medical_information_qa_graph.py` + `langgraph.json` → `uv run langgraph dev` |

**Next:** `04_deep_agent.ipynb` — delegate multi-drug inquiries to per-drug label researchers and measure structure.